# Day 29: Hybrid Search in Qdrant (Dense + Sparse Vectors)

Welcome to Day 29! Today we are moving into Phase 3: Advanced RAG & Agentic AI.

## Core Theory (Just-in-Time)

**Why Hybrid Search?**
Standard vector databases rely on dense vectors (embeddings like OpenAI's `text-embedding-3-small`) to find semantic similarity. If a user asks "Where is the company headquarters?", dense vectors easily match with "Our main office is located in Seattle."

However, dense vectors often struggle with **exact keyword matches**, acronyms, part numbers, or specific names. If a user searches for "Error code 0x80040154", a dense embedding might match generic error troubleshooting docs, completely missing the specific exact match.

**Sparse Vectors (BM25) to the Rescue**
Sparse vectors represent text based on term frequency (like BM25, TF-IDF). They are mostly zeros, with non-zero values representing specific words. They excel at exact keyword matching.

**Hybrid Search** combines both: dense vectors for meaning, and sparse vectors for keywords. Qdrant supports storing both on the same point and using **Reciprocal Rank Fusion (RRF)** to combine the results beautifully.

**AI Security Implications (PII & Prompt Injection)**
When performing hybrid search in production, you must consider security:
1.  **PII Redaction:** Before embedding queries or inserting documents, ensure sensitive data (PII like emails, SSNs) is masked or redacted. Sparse vectors (BM25) expose raw tokens, meaning unredacted PII can be easily matched and retrieved if left in the payload.
2.  **Prompt Injection & Fallbacks:** A malicious user might craft a query designed to bypass security filters. Always implement fallback mechanisms—if a query seems suspicious or fails to return results due to aggressive filtering, return a safe, default response instead of failing loudly. Ensure strict validation on any payload metadata returned.


## Code Implementation

Let's explore a tiered progression of setting up a Qdrant collection for hybrid search, inserting data, and querying it.


### Basic: Core Concept Isolated

Isolating the core setup of a hybrid Qdrant collection with minimal boilerplate.


In [1]:
import os
from typing import List, Dict, Any
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, SparseVectorParams, SparseIndexParams,
    SparseVector, PointStruct, Prefetch, FusionQuery, Fusion
)
import random

# Basic setup: In-memory client and simple hybrid collection creation
client_basic = QdrantClient(":memory:")
coll_basic = "basic_hybrid"

if not client_basic.collection_exists(coll_basic):
    client_basic.create_collection(
        collection_name=coll_basic,
        vectors_config={"dense": VectorParams(size=384, distance=Distance.COSINE)},
        sparse_vectors_config={"sparse": SparseVectorParams(index=SparseIndexParams(on_disk=False))}
    )
print("Basic hybrid collection created.")


Basic hybrid collection created.


### Medium: Clean OOP and State Management

Here we wrap the logic in a class to manage the state of the Qdrant client and the collection, demonstrating clean OOP principles.


In [2]:
class HybridSearchManager:
    def __init__(self, collection_name: str, vector_size: int = 384):
        self.client = QdrantClient(":memory:")
        self.collection_name = collection_name
        self.vector_size = vector_size
        self._setup_collection()
        
    def _setup_collection(self) -> None:
        if self.client.collection_exists(self.collection_name):
            self.client.delete_collection(self.collection_name)
            
        self.client.create_collection(
            collection_name=self.collection_name,
            vectors_config={"dense": VectorParams(size=self.vector_size, distance=Distance.COSINE)},
            sparse_vectors_config={"sparse": SparseVectorParams(index=SparseIndexParams(on_disk=False))}
        )
        print(f"Collection '{self.collection_name}' initialized.")

    def mock_dense(self) -> List[float]:
        return [random.uniform(-1.0, 1.0) for _ in range(self.vector_size)]

    def mock_sparse(self) -> SparseVector:
        indices = sorted(random.sample(range(10000), k=15))
        values = [random.uniform(0.5, 2.0) for _ in indices]
        return SparseVector(indices=indices, values=values)
        
    def insert_documents(self, docs: List[str]) -> None:
        points = [
            PointStruct(
                id=i + 1,
                vector={"dense": self.mock_dense(), "sparse": self.mock_sparse()},
                payload={"text": doc}
            ) for i, doc in enumerate(docs)
        ]
        self.client.upsert(collection_name=self.collection_name, points=points)
        print(f"Inserted {len(points)} docs.")

manager = HybridSearchManager("medium_hybrid")
manager.insert_documents(["Hello world", "Qdrant is fast", "Hybrid search is powerful"])


Collection 'medium_hybrid' initialized.
Inserted 3 docs.


### Advanced: Production-Grade Implementation

This advanced example adds strict type hinting, docstrings, PII redaction (mocked), logging, and fallback mechanisms for search failures.


In [3]:
import logging
from typing import Optional, Union
import re

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class SecureProductionHybridSearch:
    """
    Production-grade hybrid search client with PII redaction and robust error handling.
    """
    def __init__(self, collection_name: str, vector_size: int = 384):
        self.client = QdrantClient(":memory:")
        self.collection_name = collection_name
        self.vector_size = vector_size
        self._initialize_database()

    def _initialize_database(self) -> None:
        try:
            if not self.client.collection_exists(self.collection_name):
                self.client.create_collection(
                    collection_name=self.collection_name,
                    vectors_config={"dense": VectorParams(size=self.vector_size, distance=Distance.COSINE)},
                    sparse_vectors_config={"sparse": SparseVectorParams(index=SparseIndexParams(on_disk=False))}
                )
                logger.info(f"Database collection '{self.collection_name}' successfully provisioned.")
        except Exception as e:
            logger.error(f"Failed to initialize database: {e}")
            raise

    def _redact_pii(self, text: str) -> str:
        """Basic PII redaction (e.g., email addresses)."""
        return re.sub(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', '[REDACTED_EMAIL]', text)

    def _generate_dense(self) -> List[float]:
        return [random.uniform(-1.0, 1.0) for _ in range(self.vector_size)]

    def _generate_sparse(self) -> SparseVector:
        indices = sorted(random.sample(range(10000), k=10))
        values = [random.uniform(0.5, 2.0) for _ in indices]
        return SparseVector(indices=indices, values=values)

    def index_document(self, doc_id: int, text: str) -> None:
        try:
            safe_text = self._redact_pii(text)
            point = PointStruct(
                id=doc_id,
                vector={"dense": self._generate_dense(), "sparse": self._generate_sparse()},
                payload={"text": safe_text}
            )
            self.client.upsert(collection_name=self.collection_name, points=[point])
            logger.info(f"Successfully indexed document {doc_id}")
        except Exception as e:
            logger.error(f"Indexing failed for document {doc_id}: {e}")

    def secure_hybrid_search(self, query_text: str, limit: int = 3) -> List[Dict[str, Any]]:
        """
        Executes a secure hybrid search. Returns a fallback if the query fails.
        """
        try:
            # Fallback for empty/suspicious queries
            if not query_text.strip():
                logger.warning("Empty query received, returning fallback.")
                return [{"text": "Query too short or empty.", "score": 0.0}]

            safe_query = self._redact_pii(query_text)
            response = self.client.query_points(
                collection_name=self.collection_name,
                prefetch=[
                    Prefetch(query=self._generate_dense(), using="dense", limit=limit + 2),
                    Prefetch(query=self._generate_sparse(), using="sparse", limit=limit + 2),
                ],
                query=FusionQuery(fusion=Fusion.RRF),
                limit=limit,
                with_payload=True
            )
            
            results = [{"text": p.payload.get("text", ""), "score": p.score} for p in response.points]
            if not results:
                 logger.info("No results found. Returning fallback.")
                 return [{"text": "No matching documents found in the secure database.", "score": 0.0}]
            return results
            
        except Exception as e:
            logger.error(f"Search failed: {e}")
            return [{"text": "Search service temporarily unavailable.", "score": 0.0}] # Fallback mechanism

# Usage
prod_search = SecureProductionHybridSearch("prod_hybrid")
prod_search.index_document(1, "Contact admin at admin@company.com for server issues.")
results = prod_search.secure_hybrid_search("Who to contact for server issues?")
print("Search Results:")
for r in results:
    print(f"- {r['score']:.4f}: {r['text']}")


INFO:__main__:Database collection 'prod_hybrid' successfully provisioned.


INFO:__main__:Successfully indexed document 1


Search Results:
- 0.5000: Contact admin at [REDACTED_EMAIL] for server issues.


## Common Pitfalls

1. **Using Deprecated Search API:** Using the old `.search()` or `search_batch()` methods for hybrid retrieval instead of the new `query_points()` API. The old methods require complex manual chunking and client-side fusion. `query_points()` handles RRF natively on the server side.
2. **Forgetting Named Vectors:** You must explicitly define `vectors_config` as a dictionary (e.g., `{"dense": VectorParams(...) }`) to use named vectors. If you try to mix unnamed default vectors with sparse vectors, the architecture becomes confusing to manage.
3. **Inconsistent Encoders:** Using different tokenizers/models to generate the sparse embeddings during ingestion vs. retrieval. Ensure the exact same model (e.g., BM25 or SPLADE) generates the `SparseVector` objects at both steps.

## Practical Lab

**Your Task:**
Write a Python script that extends the example above. 
1. Define a Pydantic model `Document` containing `id: int`, `text: str`, and `category: str`.
2. Create a list of 5 `Document` objects about different topics.
3. Insert these documents into a new Qdrant collection named `lab_hybrid`, storing both dense and sparse vectors.
4. Implement a `query_points()` call that filters the hybrid search to only return documents where `category == 'technical'`.

*Hint: You'll need to use Qdrant's `Filter` and `FieldCondition` models inside the `query_points()` call.*

*Optional: Record a brief 2-minute async video walkthrough of your design decisions, focusing on how you managed the metadata filtering alongside the RRF query.*

In [4]:
from pydantic import BaseModel
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, SparseVectorParams, SparseIndexParams,
    SparseVector, PointStruct, Prefetch, FusionQuery, Fusion,
    Filter, FieldCondition, MatchValue
)
import random
from typing import List

class Document(BaseModel):
    id: int
    text: str
    category: str

# 1. Setup client and collection
lab_client = QdrantClient(":memory:")
lab_collection = "lab_hybrid"

lab_client.create_collection(
    collection_name=lab_collection,
    vectors_config={"dense": VectorParams(size=384, distance=Distance.COSINE)},
    sparse_vectors_config={"sparse": SparseVectorParams(index=SparseIndexParams(on_disk=False))}
)

# 2. Create 5 Document objects
docs = [
    Document(id=1, text="How to configure Nginx for reverse proxy.", category="technical"),
    Document(id=2, text="Company holiday schedule for 2024.", category="hr"),
    Document(id=3, text="Understanding Qdrant payload filters.", category="technical"),
    Document(id=4, text="Lunch menu for the cafeteria.", category="admin"),
    Document(id=5, text="Troubleshooting Python virtual environments.", category="technical"),
]

# Helper mock functions
def mock_d() -> List[float]:
    return [random.uniform(-1.0, 1.0) for _ in range(384)]

def mock_s() -> SparseVector:
    indices = sorted(random.sample(range(10000), k=10))
    values = [random.uniform(0.5, 2.0) for _ in indices]
    return SparseVector(indices=indices, values=values)

# 3. Insert documents
points = [
    PointStruct(
        id=doc.id,
        vector={"dense": mock_d(), "sparse": mock_s()},
        payload={"text": doc.text, "category": doc.category}
    )
    for doc in docs
]
lab_client.upsert(collection_name=lab_collection, points=points)
print(f"Inserted {len(points)} documents into lab collection.")

# 4. Query with filter for category == 'technical'
query_filter = Filter(
    must=[
        FieldCondition(
            key="category",
            match=MatchValue(value="technical")
        )
    ]
)

response = lab_client.query_points(
    collection_name=lab_collection,
    prefetch=[
        Prefetch(query=mock_d(), using="dense", limit=5, filter=query_filter),
        Prefetch(query=mock_s(), using="sparse", limit=5, filter=query_filter),
    ],
    query=FusionQuery(fusion=Fusion.RRF),
    limit=3,
    with_payload=True
)

print("\nFiltered Search Results (Technical Only):")
for p in response.points:
    print(f"[{p.payload.get('category')}] Score: {p.score:.4f} | Text: {p.payload.get('text')}")


Inserted 5 documents into lab collection.

Filtered Search Results (Technical Only):
[technical] Score: 0.5000 | Text: How to configure Nginx for reverse proxy.
[technical] Score: 0.3333 | Text: Understanding Qdrant payload filters.
[technical] Score: 0.2500 | Text: Troubleshooting Python virtual environments.


## Reference Links
- [Qdrant Hybrid Search Documentation](https://qdrant.tech/documentation/concepts/search/#hybrid-search)
- [Reciprocal Rank Fusion (RRF) Explained](https://qdrant.tech/articles/hybrid-search-rrf/)
- [Sparse Vectors in Qdrant](https://qdrant.tech/documentation/concepts/vectors/#sparse-vectors)
